# Tiny Transformer

This notebook builds the first GPT-style Transformer pass on top of the tokenizer and batching work.

The model receives `x` token tensors shaped `(batch_size, context_length)` and predicts the next token for every position. The target tensor `y` has the same shape, shifted one token forward.


## Colab Bootstrap

Run this cell first. If you already ran `colab_setup.ipynb` in the same Colab runtime, this will reuse that setup; if not, it installs the small dependency set and clones the repo.


In [3]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/markschatza/llm-from-scratch.git"
REPO_REF = "codex/colab-setup-workflow"
REPO_DIR = Path("/content/llm-from-scratch")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pretrain" / "tokenizer.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repo root. Run this notebook from inside the llm-from-scratch checkout."
    )

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "numpy", "torch"], check=True)
    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
    project_root = REPO_DIR
else:
    project_root = find_project_root(Path.cwd())

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project root: {project_root}")
print(f"repo ref: {REPO_REF if IN_COLAB else 'local checkout'}")
print(f"in colab: {IN_COLAB}")

project root: C:\git\LLMFromScratch
repo ref: local checkout
in colab: False


## Imports


In [4]:
from pathlib import Path

import torch

from pretrain.batching import BatchConfig, get_batch, load_token_ids, split_tokens
from pretrain.tokenizer import Tokenizer
from pretrain.transformer import GPTLanguageModel, TransformerConfig


## Prepare Token IDs

If the tokenizer output does not exist yet, create it from the small starter corpus.


In [5]:
CORPUS = Path("data/van-life-story.txt")
OUTPUT_DIR = Path("pretrain/data")
TOKEN_BIN = OUTPUT_DIR / "train.van-life-story.bin"
MODEL_PATH = OUTPUT_DIR / "sp.model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TOKEN_BIN.exists() or not MODEL_PATH.exists():
    tok = Tokenizer.train(CORPUS, vocab_size=1024, output_dir=OUTPUT_DIR, name="sp")
    all_tokens: list[int] = []
    with open(CORPUS, "r", encoding="utf-8") as fin:
        for line in fin:
            line = line.rstrip("\n")
            if line:
                all_tokens.extend(tok.encode(line))
    torch.tensor(all_tokens, dtype=torch.int64).numpy().astype("uint32").tofile(TOKEN_BIN)

print(f"token file: {TOKEN_BIN}")
print(f"tokenizer model: {MODEL_PATH}")


token file: pretrain\data\train.van-life-story.bin
tokenizer model: pretrain\data\sp.model


## Build A Batch


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

tokens = load_token_ids(TOKEN_BIN)
train_tokens, val_tokens = split_tokens(tokens, train_fraction=0.9)

batch_config = BatchConfig(batch_size=4, context_length=16, device=device)
generator = torch.Generator().manual_seed(123)
x, y = get_batch(train_tokens, batch_config, generator=generator)

print(f"x shape: {tuple(x.shape)}")
print(f"y shape: {tuple(y.shape)}")
print(f"shift check: {torch.equal(x[:, 1:], y[:, :-1])}")


device: cpu
x shape: (4, 16)
y shape: (4, 16)
shift check: True


## Instantiate The Transformer

This is intentionally tiny: enough to prove the mechanics before we scale dimensions, layers, context length, or training time.


In [7]:
tok = Tokenizer.from_pretrained(MODEL_PATH)

model_config = TransformerConfig(
    vocab_size=tok.vocab_size,
    context_length=batch_config.context_length,
    n_embd=64,
    n_head=4,
    n_layer=2,
    dropout=0.0,
)
model = GPTLanguageModel(model_config).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model_config)
print(f"parameters: {n_params:,}")


TransformerConfig(vocab_size=1024, context_length=16, n_embd=64, n_head=4, n_layer=2, dropout=0.0)
parameters: 233,216


## Forward Pass And Loss


In [8]:
logits, loss = model(x, y)

print(f"logits shape: {tuple(logits.shape)}")
print(f"loss: {loss.item():.4f}")
print(f"expected logits shape: {(batch_config.batch_size, batch_config.context_length, tok.vocab_size)}")


logits shape: (4, 16, 1024)
loss: 6.9467
expected logits shape: (4, 16, 1024)


## A Few Training Steps

This is not serious training yet. It just proves gradients, optimizer updates, and loss calculation all connect.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
train_generator = torch.Generator().manual_seed(456)

model.train()
for step in range(100):
    xb, yb = get_batch(train_tokens, batch_config, generator=train_generator)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 5 == 0 or step == 19:
        print(f"step {step:02d} | loss {loss.item():.4f}")


step 00 | loss 6.9669
step 05 | loss 6.8659
step 10 | loss 6.6925
step 15 | loss 6.4788
step 19 | loss 6.3717


## Generate A Tiny Sample


In [10]:
model.eval()
start = torch.zeros((1, 1), dtype=torch.long, device=device)
out = model.generate(start, max_new_tokens=40)
ids = out[0].tolist()

print(ids)
print()
print(tok.decode(ids))


[0, 212, 959, 428, 371, 992, 271, 995, 756, 536, 267, 729, 328, 593, 496, 453, 123, 321, 680, 14, 126, 10, 318, 973, 843, 191, 607, 359, 398, 440, 939, 899, 756, 308, 909, 550, 514, 208, 693, 632, 1021]

�i have S0it2 They or theongam way made whatyll lookse.gy�up D It--ugostful Theyes underig 1�ilehed5
